# FigURL

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import xarray as xr
import pickle
import pandas as pd

In [11]:
import figurl as fig

In [12]:
from spyglass.shijiegu.Analysis_SGU import get_linearization_map

from spyglass.shijiegu.Analysis_SGU import TrialChoice, DecodeIngredients, DecodeResults2D, ChangeofMindTheta, DecodeResultsLinear
from spyglass.shijiegu.decodeHelpers import runSessionNames

In [13]:
from non_local_detector.visualization import (
    create_interactive_2D_decoding_figurl,
)

### load decode position

In [14]:
nwb_copy_file_name = 'lewis20240109_.nwb'

In [15]:
session_interval, position_interval = runSessionNames(nwb_copy_file_name)

In [16]:
TrialChoice & {'nwb_file_name':nwb_copy_file_name}

nwb_file_name name of the NWB file,epoch the session epoch for this task and apparatus(1 based),"epoch_name session name, get from IntervalList","choice_reward pandas dataframe, choice"
lewis20240109_.nwb,2,02_Rev2Session1,=BLOB=
lewis20240109_.nwb,4,04_Rev2Session2,=BLOB=
lewis20240109_.nwb,6,06_Rev2Session3,=BLOB=
lewis20240109_.nwb,8,08_Rev2Session4,=BLOB=


In [17]:
epoch_num = 6;

In [18]:
key={'nwb_file_name':nwb_copy_file_name,'epoch':epoch_num}
print(ChangeofMindTheta & key)
log = ChangeofMindTheta().fetch1_dataframe(key)

session_name = (TrialChoice & key).fetch1('epoch_name')

print(f"Session {session_name}")

[2025-09-30 14:16:32,321][WARNING]: Skipped checksum for file with hash: 4c336741-a657-5d24-3732-2a137f916a99, and path: /stelmo/nwb/analysis/lewis20240109/lewis20240109_SO79CEQBXK.nwb


*nwb_file_name *epoch    *proportion    *delta_t_minus *delta_t_plus  *max_flag    analysis_file_
+------------+ +-------+ +------------+ +------------+ +------------+ +----------+ +------------+
lewis20240109_ 6         0.1            0              2              0            lewis20240109_
 (Total: 1)

Session 06_Rev2Session3


In [19]:
entry = DecodeIngredients & {'nwb_file_name':nwb_copy_file_name,
                             'interval_list_name':session_name}
    
# Get data
marks = xr.open_dataset(entry.fetch1('marks'))
position_1d = pd.read_csv(entry.fetch1('position_1d')) #still need 1D position
position_2d = pd.read_csv(entry.fetch1('position_2d')) # need 2D position

entry = DecodeResults2D & {'nwb_file_name':nwb_copy_file_name,
                             'interval_list_name':session_name}

environment_path = entry.fetch1('classifier')
with open(environment_path, 'rb') as file:
    environment2D = pickle.load(file)

decode_path2d = entry.fetch1('posterior')
results = xr.open_zarr(decode_path2d, consolidated=False)

#classifier_path = entry.fetch1('classifier')
#with open(classifier_path, 'rb') as file:
#    classifier = pickle.load(file)
timestamps = np.array(position_1d.time)

### Make URL

In [20]:
log[log.change_of_mind]

,timestamp_H,Home,timestamp_O,OuterWellIndex,rewardNum,change_of_mind,theta_dev,long_theta,CoMMaxProportion,initial_choice,...,proportion_arm2,proportion_arm3,proportion_arm4,CoMNum_by_time,CoMNum_by_arm,current,future_H,future_O,past,past_reward
id,,,,,,,,,,,,,,,,,,,,,
9,1.704837e+09,1.0,1.704837e+09,2.0,1.0,True,-21.453711,True,0.514749,4.0,...,0.000000,0.000000,0.514749,1,1,2.0,2.0,1.0,3.0,3.0
14,1.704837e+09,1.0,1.704837e+09,4.0,2.0,True,0.000000,False,0.828244,1.0,...,0.828244,0.000000,0.000000,3,2,4.0,4.0,1.0,3.0,2.0
23,1.704838e+09,1.0,1.704838e+09,4.0,2.0,True,-28.662976,True,0.546564,1.0,...,0.000000,0.000000,0.000000,1,1,4.0,4.0,3.0,2.0,2.0
24,1.704838e+09,1.0,1.704838e+09,3.0,2.0,True,0.000000,False,0.685085,2.0,...,0.685085,0.000000,0.284639,3,2,3.0,3.0,1.0,4.0,4.0
28,1.704838e+09,1.0,1.704838e+09,3.0,2.0,True,-35.639566,True,0.671087,1.0,...,0.000000,0.000000,0.000000,1,1,3.0,3.0,2.0,4.0,4.0
33,1.704838e+09,1.0,1.704838e+09,2.0,2.0,True,8.453178,False,0.163210,1.0,...,0.000000,0.000000,0.000000,1,1,2.0,2.0,4.0,3.0,1.0
52,1.704838e+09,1.0,1.704838e+09,4.0,2.0,True,9.509989,False,0.161185,1.0,...,0.000000,0.000000,0.000000,1,1,4.0,4.0,3.0,2.0,2.0
64,1.704839e+09,1.0,1.704839e+09,4.0,2.0,True,11.660779,False,0.156797,1.0,...,0.000000,0.000000,0.000000,1,1,4.0,4.0,1.0,3.0,2.0
69,1.704839e+09,1.0,1.704839e+09,2.0,2.0,True,12.921849,False,0.102454,3.0,...,0.000000,0.102454,0.000000,1,1,2.0,2.0,4.0,1.0,1.0


In [21]:
trialInd = 24
t0 = log.loc[trialInd,'timestamp_H']
t1 = log.loc[trialInd + 1,'timestamp_H']
if np.isnan(t0):
    t0 = t1-12

frameToPlot = np.argwhere(np.logical_and(timestamps>=t0,timestamps<=t1)).ravel()
frame0 = frameToPlot[0]
frameLast = frameToPlot[-1]

In [73]:
time_slice = slice(frame0+15000, frameLast)

In [74]:
marks

<xarray.Dataset> Size: 743MB
Dimensions:                        (time: 1119440, marks: 4, electrodes: 41)
Coordinates:
  * time                           (time) float64 9MB 1.705e+09 ... 1.705e+09
  * electrodes                     (electrodes) int32 164B 0 1 3 7 ... 61 62 63
  * marks                          (marks) object 32B 'amplitude_0000' ... 'a...
Data variables:
    __xarray_dataarray_variable__  (time, marks, electrodes) float32 734MB ...

In [75]:
####
posterior_subset = results.isel(time=time_slice)
position_subset = position_2d.iloc[time_slice]
marks_subset = marks.isel(time=time_slice)
####

In [76]:
len(marks_subset.time)

6031

array([[262.41704545,  60.22159091],
       [262.41661932,  60.22230114],
       [262.41619318,  60.22301136],
       ...,
       [211.15530303, 231.41287879],
       [211.17171717, 231.46464646],
       [211.18813131, 231.51641414]])

In [89]:
create_interactive_2D_decoding_figurl(
     position_time=position_subset.time.to_numpy(),
     position=np.array(position_subset[["head_position_x", "head_position_y"]]),
     env=environment2D,
     results=posterior_subset,
     posterior=posterior_subset.acausal_posterior.sum("state"),
     spike_times=np.array(marks_subset.time)[:10],
     head_dir=np.array(position_subset["head_orientation"]),
     speed=np.array(position_subset["head_speed"]),
)

ValueError: could not broadcast input array from shape (58440390,) into shape (1537905,)

In [78]:
#%debug

In [62]:
import os

In [63]:
os.environ['KACHERY_API_KEY'] = 'DgPXUkt8NIT3gcHG3TkNAdNdFBgC0CYS'

In [52]:
%debug

> /home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/xarray/core/common.py(285)__getattr__()
    283                 with suppress(KeyError):
    284                     return source[name]
--> 285         raise AttributeError(
    286             f"{type(self).__name__!r} object has no attribute {name!r}"
    287         )



ipdb>  u


> /home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/non_local_detector/visualization/figurl_2D.py(532)create_interactive_2D_decoding_figurl()
    530                 t=np.asarray(results.time),
    531                 y=np.asarray(
--> 532                     results.sel(state=state).acausal_state_probabilities,
    533                     dtype=np.float32,
    534                 ),



ipdb>  results.sel(state=state)


<xarray.Dataset> Size: 2GB
Dimensions:            (time: 18031, x_position: 102, y_position: 95)
Coordinates:
    state              <U10 40B 'Continuous'
  * time               (time) float64 144kB 1.705e+09 1.705e+09 ... 1.705e+09
  * x_position         (x_position) float64 816B 119.1 121.1 ... 317.3 319.2
  * y_position         (y_position) float64 760B 47.72 49.7 ... 232.4 234.4
Data variables:
    acausal_posterior  (time, x_position, y_position) float32 699MB dask.array<chunksize=(12685, 7, 6), meta=np.ndarray>
    causal_posterior   (time, x_position, y_position) float32 699MB dask.array<chunksize=(12685, 7, 6), meta=np.ndarray>
    likelihood         (time, x_position, y_position) float32 699MB dask.array<chunksize=(12685, 7, 6), meta=np.ndarray>
Attributes:
    data_log_likelihood:  -872350.1875


ipdb>  exit


In [ ]:
# from non_local_detector.visualization import (
#     create_interactive_2D_decoding_figurl,
# )

# (
#     position_info,
#     position_variable_names,
# ) = ClusterlessDecodingV1.fetch_position_info(selection_key)
# results_time = decoding_results.acausal_posterior.isel(intervals=0).time.values
# position_info = position_info.loc[results_time[0] : results_time[-1]]

# env = ClusterlessDecodingV1.fetch_environments(selection_key)[0]
# spike_times, _ = ClusterlessDecodingV1.fetch_spike_data(selection_key)


# create_interactive_2D_decoding_figurl(
#     position_time=position_info.index.to_numpy(),
#     position=position_info[position_variable_names],
#     env=env,
#     results=decoding_results,
#     posterior=decoding_results.acausal_posterior.isel(intervals=0)
#     .unstack("state_bins")
#     .sum("state"),
#     spike_times=spike_times,
#     head_dir=position_info["orientation"],
#     speed=position_info["speed"],
# )